# 🗑️ Pakistani Garbage Detection — YOLOv8 Masterclass Pipeline
> **Production-ready · End-to-end · Self-healing · Kaggle & local compatible**
> Fine-tunes YOLOv8 on 9 Pakistani waste categories with comprehensive evaluation,
> ONNX export, and zero hard-coded paths.

---

## ⚡ Quick Start
1. **Set Roboflow credentials** → `RF_API_KEY`, `RF_WORKSPACE`, `RF_PROJECT`, `RF_VERSION` in **Cell 1** *(one-time setup)*
2. **Attach dataset** *(optional)* → Kaggle sidebar ▸ *Add Data* — if attached, it takes priority over Roboflow
3. **Select GPU** → Settings ▸ Accelerator ▸ *GPU T4 x2* (or P100)
4. **Click** *Save Version → Run All* — no interruptions, no manual steps
5. **Download results** from the *Output* tab when the run finishes

> **Resuming a previous run?** Set `RESUME = True` and `RESUME_WEIGHTS` in **Cell 5 (Config)**.

---

## 📋 Table of Contents
| # | Section | Description |
|---|---------|-------------|
| 0 | [PyTorch Pre-flight](#preflight) | Install correct torch **before** any import — no restart needed |
| 1 | [Environment Setup](#env) | Kaggle paths · GPU · timestamped output dir · **Roboflow fallback** |
| 2 | [Dependencies](#deps) | Install remaining packages |
| 3 | [Imports & Logging](#imports) | All imports · logger · reproducibility seed |
| 4 | [Configuration](#config) | Single-source `TrainingConfig` dataclass |
| 5 | [Dataset Detection](#dataset) | Recursive auto-locate · tree print |
| 6 | [Dataset Validation](#validate) | Integrity check · orphan detection |
| 7 | [EDA](#eda) | Class distribution · sample images with GT boxes |
| 8 | [Training](#training) | YAML gen · callbacks · YOLOv8 train |
| 9 | [Evaluation](#eval) | mAP · per-class AP · confusion matrix |
| 10 | [Test Set](#test) | Optional test-split evaluation |
| 11 | [Export](#export) | ONNX · TensorRT (optional) · ONNX Runtime smoke test |
| 12 | [Artifacts](#artifacts) | Collect & organise outputs |
| 13 | [Summary](#summary) | Human-readable report · download instructions |

---

## 🗂️ Expected Dataset Structure
```
dataset_root/
├── train/
│   ├── images/   ← JPEG / PNG
│   └── labels/   ← YOLO-format .txt
├── valid/        ← or  val/
│   ├── images/
│   └── labels/
├── test/         ← optional
│   ├── images/
│   └── labels/
└── data.yaml     ← class names (paths rewritten by this notebook)
```

## 🏷️ Classes (9)
`Animal Waste` · `Construction Waste` · `Garbage Bag` · `Glass` · `Metal`
`Organic` · `Paper` · `Plastic` · `waste`

---

## 🛠️ Troubleshooting
| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `CUDA error: no kernel image` | PyTorch + CUDA 12.x on P100 (sm_60) | Cell 2 auto-reinstalls; restart kernel if prompted |
| `FileNotFoundError: data.yaml` | Dataset not attached | Kaggle sidebar -> Add Data |
| `RuntimeError: CUDA out of memory` | Batch too large | Set `batch_size` smaller in Cell 5 (try 8) |
| `AssertionError: train/images missing` | Non-standard layout | Print `DATASET_ROOT` and verify the tree in Cell 6 |
| ONNX export returns `None` | Ultralytics bug | Update `ultralytics` (Cell 2) and retry Cell 12 |
| TensorRT export fails | TensorRT not installed | Normal -- ONNX is always produced as fallback |
| Training curves cell errors | `results.csv` not yet written | Run Cell 9 (train) first |


---
## 0 · PyTorch Pre-flight (runs **before** any `import torch`)
Installs the correct CUDA-compatible PyTorch once, with **no kernel restart required**.
This is the key to making `Save Version` run all the way through.


In [ ]:
# ============================================================================
# CELL 0 -- PyTorch Pre-flight Installation
# MUST run before any 'import torch' statement.
# Installing torch here (while it is not yet loaded in memory) means
# the correct version is available for all subsequent cells without
# needing a kernel restart.
# ============================================================================
import subprocess, sys

TORCH_TARGET      = "2.3.1+cu118"
TORCHVISION_TARGET = "0.18.1+cu118"
INDEX_URL          = "https://download.pytorch.org/whl/cu118"

def _pip(*args):
    return subprocess.call([sys.executable, "-m", "pip"] + list(args))

needs_install = False
try:
    # Peek at the installed version WITHOUT importing the module
    import importlib.metadata as _meta
    current = _meta.version("torch")
    if current == TORCH_TARGET:
        print(f"[OK] torch {current} already installed — skipping.")
    else:
        print(f"[INFO] torch {current} detected, need {TORCH_TARGET} — reinstalling.")
        needs_install = True
except Exception:
    print("[INFO] torch not found — installing.")
    needs_install = True

if needs_install:
    CANDIDATES = [
        ("2.3.1", "0.18.1"),
        ("2.3.0", "0.18.0"),
        ("2.2.2", "0.17.2"),
        ("2.2.1", "0.17.1"),
    ]
    installed = False
    for tv, tvv in CANDIDATES:
        print(f"  Trying torch=={tv}+cu118 ...")
        rc = _pip(
            "install", "-q", "--upgrade",
            f"torch=={tv}+cu118",
            f"torchvision=={tvv}+cu118",
            "--index-url", INDEX_URL,
        )
        if rc == 0:
            print(f"[OK] torch=={tv}+cu118 installed successfully.")
            installed = True
            break
        print(f"  [WARN] {tv} unavailable, trying next ...")
    if not installed:
        raise RuntimeError(
            "Could not install a compatible PyTorch version.\n"
            "Please switch to a T4 GPU runtime and try again."
        )

# Verify the installed version is importable (sanity check)
import importlib.metadata as _meta2
print(f"[OK] torch version in site-packages: {_meta2.version('torch')}")
print("[OK] Pre-flight complete — proceeding to environment setup.")


---
## 1 · Environment Setup <a id="env"></a>
Detects Kaggle vs local, resolves I/O directories, and creates a **timestamped** output folder so successive runs never overwrite each other.

**Dataset priority:** Kaggle attached dataset → Roboflow API auto-download → local path. Set `RF_API_KEY`, `RF_WORKSPACE`, `RF_PROJECT`, and `RF_VERSION` in this cell once — the notebook will automatically pull the latest version whenever no dataset is attached.

In [ ]:
# ============================================================================
# CELL 1 -- Environment Setup + Roboflow Auto-Download (Guaranteed)
# ============================================================================
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  ROBOFLOW CONFIG  — fill these in once, never touch again               │
# │  Get your API key: https://app.roboflow.com → Settings → Roboflow API  │
# └─────────────────────────────────────────────────────────────────────────┘
RF_API_KEY   = "FPE2gtI96UzZ9KcxEjAE"   # ← paste your key here
RF_WORKSPACE = "fyp-gojku"     # ← e.g. "ali-hassan-xyz"
RF_PROJECT   = "neat-now"       # ← e.g. "pakistani-garbage"
RF_VERSION   = 5                         # ← dataset version number (int)
RF_FORMAT    = "yolov8"                  # leave as-is for YOLO training

import os, warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

RUN_TS    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
SEP       = "=" * 70
IN_KAGGLE = os.path.exists("/kaggle/input")

print(SEP)
print("ENVIRONMENT SETUP")
print(SEP)

if IN_KAGGLE:
    print("  Platform : Kaggle")
    INPUT_DIR   = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
else:
    print("  Platform : Local")
    INPUT_DIR   = Path("input")
    WORKING_DIR = Path(".")

OUTPUT_DIR = WORKING_DIR / "outputs" / RUN_TS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  Input    : {INPUT_DIR}")
print(f"  Working  : {WORKING_DIR}")
print(f"  Outputs  : {OUTPUT_DIR}")

# ── Helpers ───────────────────────────────────────────────────────────────────
_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

def _count_images(directory):
    """Return number of image files in a directory (non-recursive)."""
    p = Path(directory)
    if not p.exists():
        return 0
    return sum(1 for f in p.iterdir() if f.suffix.lower() in _IMG_EXT)

def _is_valid_yolo_root(root, min_images=10):
    """
    Return True only if `root` contains a proper YOLO layout:
      root/train/images/  with at least `min_images` files
      root/valid/ or root/val/  with at least 1 image
    This prevents system folders or empty Roboflow cache dirs from being
    silently accepted as a dataset.
    """
    root = Path(root)
    # Find train/images — may be directly under root or one level deeper
    train_candidates = list(root.rglob("train/images"))
    if not train_candidates:
        return False
    train_img = train_candidates[0]
    if _count_images(train_img) < min_images:
        return False
    base = train_img.parent.parent
    val_img = base / "valid" / "images" if (base / "valid").exists() else base / "val" / "images"
    if _count_images(val_img) < 1:
        return False
    return True

def _find_yolo_root(search_path):
    """
    Walk `search_path` and return the first directory that passes
    _is_valid_yolo_root(), or None.
    """
    search_path = Path(search_path)
    # Check the root itself first
    if _is_valid_yolo_root(search_path):
        return search_path
    # Then immediate children
    for child in sorted(search_path.iterdir()):
        if child.is_dir() and _is_valid_yolo_root(child):
            return child
    # Deep search as last resort
    for hit in search_path.rglob("train/images"):
        candidate = hit.parent.parent
        if _is_valid_yolo_root(candidate):
            return candidate
    return None

# ── Dataset Resolution ────────────────────────────────────────────────────────
DATASET_ROOT   = None
DATASET_SOURCE = None   # "kaggle" | "roboflow" | "local"

# ── Strategy A: Kaggle attached dataset ──────────────────────────────────────
if IN_KAGGLE:
    available_datasets = sorted(INPUT_DIR.iterdir()) if INPUT_DIR.exists() else []
    print(f"\nAttached Kaggle datasets ({len(available_datasets)}):")
    for ds in available_datasets:
        print(f"  * {ds.name}")

    for ds in available_datasets:
        root = _find_yolo_root(ds)
        if root is not None:
            DATASET_ROOT   = root
            DATASET_SOURCE = "kaggle"
            n_train = _count_images(next(root.rglob("train/images")))
            print(f"  [OK] Valid YOLO dataset found: {ds.name}  ({n_train} train images)")
            break

    if DATASET_ROOT is None and available_datasets:
        print("  [WARN] Attached dataset(s) found but none passed YOLO structure check.")
        print("         Falling through to Roboflow download.")

# ── Strategy B: Roboflow auto-download ───────────────────────────────────────
if DATASET_ROOT is None:
    if not IN_KAGGLE:
        print("\n  No Kaggle environment — checking Roboflow ...")
    else:
        print("\n  Falling back to Roboflow auto-download ...")

    import subprocess, sys as _sys

    # Install roboflow if not already present
    try:
        import roboflow as _rf_chk  # noqa
    except ImportError:
        print("  [RF] Installing roboflow package ...")
        subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", "roboflow"])

    from roboflow import Roboflow

    if not RF_API_KEY or RF_API_KEY == "YOUR_ROBOFLOW_API_KEY":
        raise ValueError(
            "\nRF_API_KEY is not set!\n"
            "→ Open Cell 1 and paste your Roboflow API key into RF_API_KEY.\n"
            "  Get it at: https://app.roboflow.com → Settings → Roboflow API"
        )

    dl_dir = WORKING_DIR / "rf_dataset"
    dl_dir.mkdir(parents=True, exist_ok=True)

    # Smart cache: only skip download if folder is already valid
    # (fixes the silent-corrupt-cache bug from overwrite=False)
    cached_root = _find_yolo_root(dl_dir)
    if cached_root is not None:
        n_cached = _count_images(next(cached_root.rglob("train/images")))
        print(f"  [RF] Valid cached dataset found at {cached_root}  ({n_cached} train images) — skipping download.")
        DATASET_ROOT   = cached_root
        DATASET_SOURCE = "roboflow-cache"
    else:
        print(f"  [RF] Connecting  → workspace: {RF_WORKSPACE}  project: {RF_PROJECT}  v{RF_VERSION}")
        try:
            rf      = Roboflow(api_key=RF_API_KEY)
            project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
            version = project.version(RF_VERSION)

            print(f"  [RF] Downloading v{RF_VERSION} ({RF_FORMAT}) → {dl_dir} ...")
            # overwrite=True ensures a fresh download — we handle caching ourselves above
            dataset = version.download(RF_FORMAT, location=str(dl_dir), overwrite=True)

            # dataset.location is where Roboflow placed the files;
            # it may be dl_dir itself or a subfolder like dl_dir/project-v1/
            raw_loc = Path(dataset.location)
            dl_root = _find_yolo_root(raw_loc)
            if dl_root is None:
                # Roboflow sometimes nests one level up
                dl_root = _find_yolo_root(dl_dir)
            if dl_root is None:
                raise RuntimeError(
                    f"Download reported success but no valid YOLO structure found under {dl_dir}.\n"
                    f"Contents: {list(dl_dir.iterdir())}"
                )

            n_dl = _count_images(next(dl_root.rglob("train/images")))
            print(f"  [RF] Download verified — {n_dl} train images found at {dl_root}")
            DATASET_ROOT   = dl_root
            DATASET_SOURCE = "roboflow"

        except Exception as _rf_err:
            raise RuntimeError(
                f"\nRoboflow download failed: {_rf_err}\n\n"
                "Fix options:\n"
                "  1. Attach your dataset via the Kaggle sidebar (Add Data).\n"
                "  2. Check RF_API_KEY / RF_WORKSPACE / RF_PROJECT / RF_VERSION in Cell 1.\n"
                "  3. Verify the project exists at https://app.roboflow.com"
            ) from _rf_err

# ── Strategy C: Local path (non-Kaggle only) ─────────────────────────────────
if not IN_KAGGLE and DATASET_ROOT is None:
    local_path = Path("FYP/final_dataset")
    if _is_valid_yolo_root(local_path):
        DATASET_ROOT   = local_path
        DATASET_SOURCE = "local"
    else:
        raise RuntimeError(
            f"Local dataset not found or invalid at {local_path}.\n"
            "Set RF_API_KEY and Roboflow credentials above to auto-download."
        )

# ── Final guard ───────────────────────────────────────────────────────────────
assert DATASET_ROOT is not None, "DATASET_ROOT was never set — all strategies failed."
assert DATASET_ROOT.exists(),    f"DATASET_ROOT does not exist: {DATASET_ROOT}"
assert _is_valid_yolo_root(DATASET_ROOT), (
    f"DATASET_ROOT exists but failed YOLO structure check: {DATASET_ROOT}\n"
    f"Expected: train/images/ with images  +  valid/images/ with images."
)

_train_n = _count_images(next(DATASET_ROOT.rglob("train/images")))
_val_candidates = list(DATASET_ROOT.rglob("valid/images")) or list(DATASET_ROOT.rglob("val/images"))
_val_dir        = _val_candidates[0] if _val_candidates else None
_val_n    = _count_images(_val_dir) if _val_dir else 0

print(f"\n[OK] Dataset root   : {DATASET_ROOT}")
print(f"[OK] Dataset source : {DATASET_SOURCE}")
print(f"[OK] Train images   : {_train_n:,}")
print(f"[OK] Val   images   : {_val_n:,}")
print(SEP)


In [ ]:
# ============================================================================
# CELL 1B — Construction Waste Augmentation + Kaggle Dataset Upload
# ============================================================================
# INSERT THIS CELL IMMEDIATELY AFTER CELL 1 (Roboflow download)
#           and BEFORE CELL 6 (dataset detection / path resolution).
#
# Requires: DATASET_ROOT, WORKING_DIR  (both set by Cell 1)
# Effect  : Writes new images + labels directly into train/images and
#           train/labels of the already-downloaded dataset so that Cell 6
#           picks them up automatically with no path changes.
# ============================================================================

import os, shutil, random, json as _json, subprocess
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# USER TOGGLES — the only lines you ever need to change
# ─────────────────────────────────────────────────────────────────────────────
AUGMENT_CONSTRUCTION_WASTE  = True   # False → skip augmentation entirely
UPLOAD_AUGMENTED_TO_KAGGLE  = True   # False → skip Kaggle upload

CONSTRUCTION_WASTE_CLASS_ID = 1      # YOLO class index for Construction Waste
COPIES_PER_IMAGE            = 4      # augmented copies per qualifying train image
AUG_SEED                    = 42     # reproducibility seed

# Kaggle dataset settings (only used when UPLOAD_AUGMENTED_TO_KAGGLE=True)
KAGGLE_DATASET_TITLE        = "Garbage Augmented v1"
KAGGLE_DATASET_SLUG         = "garbage-augmented-v1"   # URL slug — no spaces
# ─────────────────────────────────────────────────────────────────────────────

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
_SEP      = "=" * 70


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — AUGMENTATION                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
if AUGMENT_CONSTRUCTION_WASTE:
    print("\n" + _SEP)
    print("CONSTRUCTION WASTE AUGMENTATION")
    print(_SEP)

    # ── Install albumentations if missing ─────────────────────────────────────
    try:
        import albumentations as A
    except ImportError:
        print("[INFO] Installing albumentations ...")
        subprocess.run(["pip", "install", "-q", "albumentations"], check=True)
        import albumentations as A

    import cv2
    import numpy as np

    # ── Resolve train split paths from DATASET_ROOT ───────────────────────────
    # Cell 6 hasn't run yet, so we locate paths ourselves using the same
    # rglob strategy.
    def _find_train_split(root):
        for hit in Path(root).rglob("train/images"):
            img_dir = hit
            lbl_dir = hit.parent.parent / "train" / "labels"
            lbl_dir.mkdir(parents=True, exist_ok=True)
            return img_dir, lbl_dir
        return None, None

    _tr_img, _tr_lbl = _find_train_split(DATASET_ROOT)
    if _tr_img is None:
        print("[WARN] Could not locate train/images under DATASET_ROOT. Skipping.")
    else:
        print(f"  Train images dir : {_tr_img}")
        print(f"  Train labels dir : {_tr_lbl}")

        # ── Albumentations pipeline ───────────────────────────────────────────
        # All transforms are bbox-aware. min_visibility=0.3 discards any
        # augmented copy where a box loses more than 70 % of its area (e.g.
        # after a large rotation crops the box), keeping only clean annotations.
        _aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.ShiftScaleRotate(
                shift_limit=0.10, scale_limit=0.30, rotate_limit=20,
                border_mode=cv2.BORDER_REFLECT_101, p=0.70,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.35, contrast_limit=0.35, p=0.70,
            ),
            A.HueSaturationValue(
                hue_shift_limit=15, sat_shift_limit=35, val_shift_limit=25, p=0.50,
            ),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.40),
            A.OneOf([
                A.MotionBlur(blur_limit=5),
                A.MedianBlur(blur_limit=3),
                A.GaussianBlur(blur_limit=3),
            ], p=0.25),
            A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.30),
            A.RandomGamma(gamma_limit=(80, 120), p=0.30),
        ], bbox_params=A.BboxParams(
            format        = "yolo",         # (cx, cy, w, h) normalised 0-1
            label_fields  = ["class_labels"],
            min_visibility= 0.30,           # drop boxes that become <30 % visible
            clip          = True,           # clamp boxes to image boundary
        ))

        # ── Label helpers ─────────────────────────────────────────────────────
        def _has_class(lbl_path, cid):
            lp = Path(lbl_path)
            if not lp.exists():
                return False
            for line in lp.read_text().splitlines():
                parts = line.strip().split()
                if parts and int(parts[0]) == cid:
                    return True
            return False

        def _read_labels(lbl_path):
            labels = []
            for line in Path(lbl_path).read_text().splitlines():
                p = line.strip().split()
                if len(p) == 5:
                    labels.append((int(p[0]), float(p[1]), float(p[2]),
                                   float(p[3]), float(p[4])))
            return labels

        def _write_labels(lbl_path, labels):
            lines = [f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                     for c, cx, cy, w, h in labels]
            Path(lbl_path).write_text("\n".join(lines))

        # ── Collect qualifying images (train only) ────────────────────────────
        all_train = sorted(
            f for f in _tr_img.iterdir() if f.suffix.lower() in _IMG_EXTS
        )
        # Only augment images that actually contain Construction Waste labels.
        # Images with other classes present are also augmented (all their boxes
        # are carried over correctly), which is intentional: mixing classes
        # helps the model learn co-occurrence context.
        qualify = [
            f for f in all_train
            if _has_class(_tr_lbl / f"{f.stem}.txt", CONSTRUCTION_WASTE_CLASS_ID)
        ]

        print(f"\n  Train images total       : {len(all_train):,}")
        print(f"  With Construction Waste  : {len(qualify):,}  (class {CONSTRUCTION_WASTE_CLASS_ID})")
        print(f"  Copies per image         : {COPIES_PER_IMAGE}")
        print(f"  Expected new images      : up to {len(qualify) * COPIES_PER_IMAGE:,}\n")

        total_created = skipped = 0
        for idx, img_path in enumerate(qualify):
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                print(f"  [SKIP] Cannot read: {img_path.name}")
                skipped += 1
                continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            labels  = _read_labels(_tr_lbl / f"{img_path.stem}.txt")
            if not labels:
                skipped += 1
                continue

            bboxes  = [(cx, cy, w, h) for _, cx, cy, w, h in labels]
            cls_ids = [c for c, *_ in labels]

            for k in range(COPIES_PER_IMAGE):
                # Per-copy deterministic seed for exact reproducibility
                random.seed(AUG_SEED + idx * 100 + k)
                np.random.seed(AUG_SEED + idx * 100 + k)

                try:
                    out = _aug(image=img_rgb, bboxes=bboxes, class_labels=cls_ids)
                except Exception as e:
                    continue   # rare: skip this copy silently

                # Skip if ALL boxes were lost (fully outside image after transform)
                if not out["bboxes"]:
                    continue

                out_stem = f"{img_path.stem}_cw_aug{k:02d}"
                cv2.imwrite(
                    str(_tr_img / f"{out_stem}{img_path.suffix}"),
                    cv2.cvtColor(out["image"], cv2.COLOR_RGB2BGR),
                )
                _write_labels(
                    _tr_lbl / f"{out_stem}.txt",
                    [(c, cx, cy, w, h)
                     for c, (cx, cy, w, h)
                     in zip(out["class_labels"], out["bboxes"])],
                )
                total_created += 1

            # Progress update every 50 images
            if (idx + 1) % 50 == 0 or (idx + 1) == len(qualify):
                print(f"  [{idx+1:>4}/{len(qualify)}] augmented images so far: {total_created}")

        new_total = len(list(_tr_img.iterdir()))
        print(f"\n[OK] Augmentation complete.")
        print(f"     New augmented images  : {total_created}")
        print(f"     Skipped (unreadable)  : {skipped}")
        print(f"     Train images now      : {new_total:,}  (was {len(all_train):,})")
        print(_SEP)


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 2 — KAGGLE DATASET UPLOAD                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
if UPLOAD_AUGMENTED_TO_KAGGLE:
    print("\n" + _SEP)
    print("KAGGLE DATASET UPLOAD")
    print(_SEP)

    # ── 1. Resolve Kaggle credentials ─────────────────────────────────────────
    # Priority: environment variables > Kaggle Secrets (notebook add-on)
    _kag_user = os.environ.get("KAGGLE_USERNAME", "")
    _kag_key  = os.environ.get("KAGGLE_KEY",      "")

    if not (_kag_user and _kag_key):
        try:
            from kaggle_secrets import UserSecretsClient
            _sec = UserSecretsClient()
            if not _kag_user:
                _kag_user = _sec.get_secret("KAGGLE_USERNAME")
            if not _kag_key:
                _kag_key  = _sec.get_secret("KAGGLE_KEY")
        except Exception as _e:
            print(f"  [WARN] Could not load Kaggle Secrets: {_e}")

    if not (_kag_user and _kag_key):
        print(
            "[ERROR] Kaggle credentials not found.\n"
            "        Add them via:  Notebook top-bar → Add-ons → Secrets\n"
            "        Required secret names: KAGGLE_USERNAME, KAGGLE_KEY\n"
            "        Get your key at: https://www.kaggle.com/settings → API"
        )
    else:
        # ── 2. Write ~/.kaggle/kaggle.json ────────────────────────────────────
        _kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        _kaggle_json.parent.mkdir(exist_ok=True)
        _kaggle_json.write_text(
            _json.dumps({"username": _kag_user, "key": _kag_key})
        )
        os.chmod(_kaggle_json, 0o600)
        print(f"  [OK] Kaggle credentials configured for user: {_kag_user}")

        # ── 3. Stage a clean copy of the dataset ─────────────────────────────
        # We copy rather than upload DATASET_ROOT in-place so we can safely
        # add dataset-metadata.json without modifying the live training data.
        _stage = WORKING_DIR / "_augmented_stage"
        if _stage.exists():
            shutil.rmtree(_stage)
        print(f"  [INFO] Staging dataset copy → {_stage}")
        print(f"         (This copies {DATASET_ROOT} — may take a minute for large datasets)")
        shutil.copytree(str(DATASET_ROOT), str(_stage))
        print(f"  [OK] Stage ready.")

        # ── 4. Write dataset-metadata.json ────────────────────────────────────
        _meta = {
            "title":    KAGGLE_DATASET_TITLE,
            "id":       f"{_kag_user}/{KAGGLE_DATASET_SLUG}",
            "licenses": [{"name": "CC0-1.0"}],
        }
        (_stage / "dataset-metadata.json").write_text(_json.dumps(_meta, indent=2))
        print(f"  [INFO] Dataset id : {_meta['id']}")
        print(f"  [INFO] Title      : {_meta['title']}")

        # ── 5. Create or version the Kaggle dataset ───────────────────────────
        def _kaggle_run(cmd):
            r = subprocess.run(cmd, capture_output=True, text=True)
            return r.returncode, (r.stdout + r.stderr).strip()

        print("\n  [INFO] Attempting to create new dataset ...")
        _rc, _out = _kaggle_run([
            "kaggle", "datasets", "create",
            "-p", str(_stage), "--dir-mode", "zip",
        ])

        _slug_url = f"https://www.kaggle.com/datasets/{_kag_user}/{KAGGLE_DATASET_SLUG}"

        if _rc == 0:
            print(f"\n  [OK] Dataset created (private):")
            print(f"       {_slug_url}")
        elif "already exists" in _out.lower() or "409" in _out:
            print("  [INFO] Dataset already exists — creating a new version ...")
            _rc2, _out2 = _kaggle_run([
                "kaggle", "datasets", "version",
                "-p", str(_stage),
                "-m", f"Augmented Construction Waste — {COPIES_PER_IMAGE}× copies",
                "--dir-mode", "zip",
            ])
            if _rc2 == 0:
                print(f"\n  [OK] New version created:")
                print(f"       {_slug_url}")
            else:
                print(f"\n  [ERROR] Version creation failed (exit {_rc2}):")
                print(f"          {_out2}")
        else:
            print(f"\n  [ERROR] Dataset creation failed (exit {_rc}):")
            print(f"          {_out}")
            print(
                "\n  Troubleshooting:\n"
                "   • Check KAGGLE_USERNAME and KAGGLE_KEY secrets are correct.\n"
                "   • Ensure your account has 'Create Public Datasets' permission\n"
                "     disabled if you want it private (Kaggle requires Pro for private).\n"
                "   • Large datasets (>5 GB zipped) may time out; consider uploading\n"
                "     only train/images + train/labels in that case."
            )

        print(
            f"\n  To attach in future runs:\n"
            f"    Notebook sidebar → Add Data → My Datasets\n"
            f"    → search  \"{KAGGLE_DATASET_TITLE}\"  → Add\n"
            f"    The notebook's Cell 1 will then find it automatically\n"
            f"    via Strategy A (Kaggle attached dataset) and skip Roboflow."
        )
        print(_SEP)

### 1b · GPU & Hardware Check
Auto-tunes recommended batch size and image size from detected VRAM, and flags Pascal-GPU incompatibility with the current PyTorch build.

In [ ]:
# ============================================================================
# CELL 2 -- GPU Detection + CUDA Compatibility Check  (P100-optimised)
# ============================================================================
import subprocess, sys
import torch

SEP = '=' * 70
print('\n' + SEP + '\nGPU & RUNTIME CHECK\n' + SEP)

CUDA_AVAILABLE        = torch.cuda.is_available()
NEEDS_TORCH_REINSTALL = False
DEVICE                = 'cpu'
GPU_NAME              = 'CPU'
VRAM_GB               = 0.0
RECOMMENDED_BATCH     = 4
RECOMMENDED_IMGSZ     = 416
AMP_SUPPORTED         = False
IS_P100               = False

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {CUDA_AVAILABLE}')

if CUDA_AVAILABLE:
    DEVICE = 'cuda'
    props  = torch.cuda.get_device_properties(0)
    GPU_NAME  = props.name
    VRAM_GB   = props.total_memory / 1024**3
    CC_MAJOR  = props.major
    CC_MINOR  = props.minor
    CC        = float(f'{CC_MAJOR}.{CC_MINOR}')

    # P100 = Pascal architecture, compute 6.0/6.1
    IS_P100   = 'P100' in GPU_NAME or CC_MAJOR == 6

    # AMP: Pascal supports FP16 but NOT BF16
    AMP_SUPPORTED = True

    print(f'\nGPU  : {GPU_NAME}')
    print(f'VRAM : {VRAM_GB:.2f} GB')
    print(f'Compute Capability : {CC_MAJOR}.{CC_MINOR}')

    cuda_kernel_ok = False
    try:
        _ = torch.zeros(1, device='cuda') + 1
        cuda_kernel_ok = True
    except Exception:
        pass

    if cuda_kernel_ok:
        print('[OK] CUDA kernel smoke-test passed')
    else:
        print('[WARN] CUDA kernel smoke-test failed.')

    if VRAM_GB >= 15:
        RECOMMENDED_BATCH, RECOMMENDED_IMGSZ, tier = 32, 640, 'High-end (>=15 GB)'
    elif VRAM_GB >= 10:
        RECOMMENDED_BATCH, RECOMMENDED_IMGSZ, tier = 16, 640, 'Mid-range (10-15 GB)'
    elif VRAM_GB >= 6:
        RECOMMENDED_BATCH, RECOMMENDED_IMGSZ, tier = 8,  640, 'Entry (6-10 GB)'
    else:
        RECOMMENDED_BATCH, RECOMMENDED_IMGSZ, tier = 4,  416, 'Low VRAM (<6 GB)'

    if IS_P100:
        print('\n[P100 DETECTED] Applying Pascal-specific optimisations:')
        print('  half=True  -- FP16 (P100 has excellent FP16 throughput)')
        print('  amp=True   -- mixed precision via FP16, not BF16')
        print('  cache=ram  -- cache dataset in RAM (16 GB VRAM + 13 GB RAM on Kaggle)')
        print('  workers=4  -- Kaggle P100 nodes have 4 CPU cores')
        print('  optimizer=SGD -- stable on Pascal; AdamW can OOM on FP16')

    print(f'\nTier              : {tier}')
else:
    print('\n[WARN] No GPU -- training will be very slow.')

print(f'Recommended batch : {RECOMMENDED_BATCH}')
print(f'Recommended imgsz : {RECOMMENDED_IMGSZ}')
print(f'P100 mode         : {IS_P100}')
print(SEP)


### 1c · Install Dependencies

In [ ]:
# ============================================================================
# CELL 3 -- Dependency Installation + PyTorch/CUDA Auto-fix
# ============================================================================
import subprocess, sys, importlib

def _run(cmd):
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    return proc.returncode

# torch was pre-installed in CELL 0 (before any import) — no restart needed.
print("[OK] torch pre-installed in Cell 0 — skipping reinstall block.")

print("\nInstalling / upgrading dependencies ...")
_run(
    f"{sys.executable} -m pip install -q -U "
    "ultralytics albumentations opencv-python-headless "
    "matplotlib seaborn pandas scikit-learn tqdm onnx onnxruntime roboflow"
)

REQUIRED = {
    "ultralytics":    "ultralytics",
    "cv2":            "opencv-python-headless",
    "albumentations": "albumentations",
    "onnx":           "onnx",
    "onnxruntime":    "onnxruntime",
    "roboflow":       "roboflow",
}
all_ok = True
for module, pkg in REQUIRED.items():
    try:
        m   = importlib.import_module(module)
        ver = getattr(m, "__version__", "?")
        print(f"  [OK] {pkg:<32} {ver}")
    except ImportError:
        print(f"  [FAIL] {pkg} not found")
        all_ok = False

import torch
print(f"  [OK] {'torch':<32} {torch.__version__}")
try:
    _ = torch.zeros(1, device="cuda") + 1
    print("  [OK] CUDA smoke-test           passed")
except Exception as e:
    print(f"  [WARN] CUDA smoke-test: {e}")

print("\n[OK] All dependencies ready." if all_ok else "\n[WARN] Fix issues above before training.")


---
## 2 · Imports & Logging <a id="imports"></a>

In [ ]:
# ============================================================================
# CELL 4 -- All Imports, Logging, and Reproducibility Seed
# ============================================================================
import csv
import json
import logging
import random
import shutil
import textwrap
import time
from collections import Counter
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from tqdm import tqdm
from ultralytics import YOLO

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})
sns.set_theme(style="whitegrid", palette="muted")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Logger
LOG_FILE = OUTPUT_DIR / f"training_{RUN_TS}.log"

def get_logger(name="waste_det"):
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "[%(asctime)s] %(levelname)-8s -- %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    fh = logging.FileHandler(LOG_FILE)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

log = get_logger()
log.info("Logger ready -> %s", LOG_FILE)
log.info("Seed=%d", SEED)
print(f"[OK] All imports done  |  log -> {LOG_FILE}")


---
## 3 · Centralised Configuration <a id="config"></a>

All hyperparameters live in `TrainingConfig`. Edit values here — nowhere else.

| Key flag | Default | Notes |
|----------|---------|-------|
| `model_name` | `yolov8m.pt` | n/s/m/l/x variants |
| `epochs` | `100` | training budget |
| `batch_size` | auto | overridden by GPU tier |
| `patience` | `30` | early-stop epochs |
| `RESUME` | `False` | set `True` + path to resume |
| `RUN_EDA` | `True` | set `False` to skip EDA |

In [ ]:
# ============================================================================
# CELL 5 -- Centralised Configuration  (edit here only)
# ============================================================================

# User toggles
RESUME         = False   # True  = resume from RESUME_WEIGHTS
RESUME_WEIGHTS = ""      # e.g.  "/kaggle/working/outputs/.../best.pt"
RUN_EDA        = True    # False = skip EDA grid
RUN_TEST_EVAL  = True    # False = skip test-split evaluation

@dataclass
class TrainingConfig:
    # Model
    model_name: str  = "yolov8m.pt"
    # Training
    epochs: int      = 100
    img_size: int    = 640
    batch_size: int  = 16
    lr0: float       = 0.01
    lrf: float       = 0.01
    momentum: float  = 0.937
    weight_decay: float = 0.0005
    warmup_epochs: int  = 3
    patience: int    = 30
    workers: int     = 4
    amp: bool        = True
    # Augmentation
    hsv_h: float     = 0.015
    hsv_s: float     = 0.700
    hsv_v: float     = 0.400
    degrees: float   = 10.0
    translate: float = 0.10
    scale: float     = 0.50
    shear: float     = 2.0
    perspective: float = 0.0
    flipud: float    = 0.00
    fliplr: float    = 0.50
    mosaic: float    = 1.00
    mixup: float     = 0.07
    copy_paste: float = 0.07
    # Output
    project_name: str    = "waste_detection"
    experiment_name: str = "yolov8m_garbage_v1"
    # Classes
    class_names: List[str] = field(default_factory=lambda: [
        "Animal Waste", "Construction Waste", "Garbage Bag",
        "Glass", "Metal", "Organic", "Paper", "Plastic", "waste",
    ])

    @property
    def num_classes(self):
        return len(self.class_names)

CFG = TrainingConfig()
if CFG.batch_size == -1:
    CFG.batch_size = RECOMMENDED_BATCH
CFG.img_size = RECOMMENDED_IMGSZ

print("TrainingConfig")
print("=" * 70)
for k, v in asdict(CFG).items():
    print(f"  {k:<22} : {v}")
print(f"  {'num_classes':<22} : {CFG.num_classes}")
print("=" * 70)
print(f"Resume mode   : {RESUME}")
print(f"Run EDA       : {RUN_EDA}")
print(f"Run test eval : {RUN_TEST_EVAL}")


---
## 4 · Dataset Auto-Detection <a id="dataset"></a>
Recursively scans for the YOLO layout (`train/images`, `valid/images`) regardless of nesting depth.

In [ ]:
# ============================================================================
# CELL 6 -- Dataset Path Detection (Recursive, Multi-Strategy)
# ============================================================================
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
SEP = "=" * 70

print("\n" + SEP + "\nDATASET PATH DETECTION\n" + SEP)

# Print tree
print(f"\nScanning: {DATASET_ROOT}")
for root_dir, dirs, files in os.walk(DATASET_ROOT):
    depth = len(Path(root_dir).relative_to(DATASET_ROOT).parts)
    if depth > 4:
        dirs.clear()
        continue
    indent = "  " * depth
    print(f"{indent}[DIR] {Path(root_dir).name}/")
    if depth <= 2:
        shown = 0
        for f in sorted(files):
            if shown >= 5:
                print(f"{indent}  ... +{len(files)-shown} more")
                break
            print(f"{indent}  {f}")
            shown += 1
print()

def find_yolo_paths(root):
    root = Path(root)
    candidates = list(root.rglob("train/images"))
    if not candidates:
        for p in root.rglob("train"):
            if p.is_dir() and any(f.suffix.lower() in IMG_EXTENSIONS for f in p.glob("*")):
                candidates.append(p.parent / "images")
                break
    if not candidates:
        return None

    train_img = candidates[0]
    base = train_img.parent.parent
    val_name = "valid" if (base / "valid").exists() else "val"

    def _pair(split):
        img_d = base / split / "images"
        lbl_d = base / split / "labels"
        if img_d.exists():
            return img_d, (lbl_d if lbl_d.exists() else img_d)
        return None, None

    tr_img,  tr_lbl  = _pair("train")
    val_img, val_lbl = _pair(val_name)
    tst_img, tst_lbl = _pair("test")
    return {
        "train_img": tr_img, "train_lbl": tr_lbl,
        "val_img":   val_img, "val_lbl":   val_lbl,
        "test_img":  tst_img, "test_lbl":  tst_lbl,
    }

paths = find_yolo_paths(DATASET_ROOT)
if paths is None:
    raise RuntimeError("Dataset layout not recognised. Expected: root/train/images + root/valid/images")

TRAIN_IMAGES = paths["train_img"]
TRAIN_LABELS = paths["train_lbl"]
VAL_IMAGES   = paths["val_img"]
VAL_LABELS   = paths["val_lbl"]
TEST_IMAGES  = paths["test_img"]
TEST_LABELS  = paths["test_lbl"]

assert TRAIN_IMAGES and TRAIN_IMAGES.exists(), f"train/images not found: {TRAIN_IMAGES}"
assert VAL_IMAGES   and VAL_IMAGES.exists(),   f"val/images not found: {VAL_IMAGES}"

def count_images(directory):
    if directory is None or not Path(directory).exists():
        return 0
    return sum(1 for f in Path(directory).glob("*") if f.suffix.lower() in IMG_EXTENSIONS)

print("[OK] Paths resolved:\n")
for label, p in [
    ("Train Images", TRAIN_IMAGES), ("Train Labels", TRAIN_LABELS),
    ("Val   Images", VAL_IMAGES),   ("Val   Labels", VAL_LABELS),
    ("Test  Images", TEST_IMAGES),  ("Test  Labels", TEST_LABELS),
]:
    if p and p.exists():
        n = sum(1 for f in p.glob("*") if f.is_file())
        print(f"  [OK] {label:<16}: {p}  [{n} files]")
    else:
        print(f"  [--] {label:<16}: NOT FOUND")

log.info("Dataset paths OK -- train=%s  val=%s", TRAIN_IMAGES, VAL_IMAGES)
print(SEP)


---
## 5 · Dataset Integrity Validation <a id="validate"></a>
Checks every image has a label, every label has an image, class IDs are in range, and annotation format is correct.

In [ ]:
# ============================================================================
# CELL 7 -- Dataset Integrity Validation  (fast, segmentation-aware)
# ============================================================================
# Set False to skip entirely and save 30-60 seconds on large datasets
VALIDATE_DATASET = True

SEP = '=' * 70
print('\n' + SEP + '\nDATASET INTEGRITY VALIDATION\n' + SEP)

n_train = count_images(TRAIN_IMAGES)
n_val   = count_images(VAL_IMAGES)
n_test  = count_images(TEST_IMAGES) if TEST_IMAGES else 0

print(f'\nSplit sizes:')
print(f'  Train : {n_train:>6,} images')
print(f'  Val   : {n_val:>6,} images')
print(f'  Test  : {n_test:>6,} images  {"(not found)" if not n_test else ""}')
print(f'  Total : {n_train + n_val + n_test:>6,} images')

if not VALIDATE_DATASET:
    print('\n[SKIP] VALIDATE_DATASET=False -- skipping label checks.')
    print(SEP)
else:
    import concurrent.futures, time

    def _check_label_file(args):
        lbl_path, split_name, num_classes = args
        bad_fmt = seg_fmt = inv_cls = 0
        cls_counts = {}
        try:
            content = lbl_path.read_text().strip()
            if not content:
                return (lbl_path.stem, 0, 0, 0, {}, f'[{split_name}] Empty: {lbl_path.name}')
            for line in content.splitlines():
                parts = line.strip().split()
                if not parts:
                    continue
                n = len(parts)
                if n == 5:
                    cls_id = int(parts[0])
                    if not (0 <= cls_id < num_classes):
                        inv_cls += 1
                    else:
                        cls_counts[cls_id] = cls_counts.get(cls_id, 0) + 1
                elif n > 5 and n % 2 == 1:
                    seg_fmt += 1  # segmentation polygon -- not an error
                else:
                    bad_fmt += 1
        except Exception as exc:
            return (lbl_path.stem, 0, 0, 0, {}, f'[{split_name}] Read error {lbl_path.name}: {exc}')
        return (lbl_path.stem, bad_fmt, seg_fmt, inv_cls, cls_counts, None)

    def validate_split_fast(img_dir, lbl_dir, split_name, num_classes):
        img_stems = {f.stem for f in img_dir.glob('*') if f.suffix.lower() in IMG_EXTENSIONS}
        lbl_stems  = {f.stem for f in lbl_dir.glob('*.txt')}
        missing_lbl = len(img_stems - lbl_stems)
        orphan_lbl  = len(lbl_stems - img_stems)
        lbl_files = list(lbl_dir.glob('*.txt'))
        args = [(p, split_name, num_classes) for p in lbl_files]
        total_bad = total_seg = total_inv = total_err = 0
        cls_totals = Counter()
        bad_examples = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
            for stem, bad, seg, inv, cls_d, err in ex.map(_check_label_file, args):
                if err:
                    total_err += 1
                    if len(bad_examples) < 5:
                        bad_examples.append(err)
                    continue
                total_bad += bad
                total_seg += seg
                total_inv += inv
                for k, v in cls_d.items():
                    cls_totals[k] += v
                if (bad or inv) and len(bad_examples) < 5:
                    bad_examples.append(f'[{split_name}] {stem}: {bad} malformed, {inv} bad class ID')
        return dict(missing_labels=missing_lbl, orphan_labels=orphan_lbl,
                    bad_format=total_bad, seg_format=total_seg,
                    invalid_cls=total_inv, read_errors=total_err,
                    cls_counts=cls_totals, examples=bad_examples)

    t0 = time.perf_counter()
    train_r = validate_split_fast(TRAIN_IMAGES, TRAIN_LABELS, 'train', CFG.num_classes)
    val_r   = validate_split_fast(VAL_IMAGES,   VAL_LABELS,   'val',   CFG.num_classes)
    elapsed = time.perf_counter() - t0

    def _print_split(name, r):
        print(f'\n  [{name}]')
        print(f'    Missing labels    : {r["missing_labels"]}')
        print(f'    Orphan labels     : {r["orphan_labels"]}')
        print(f'    Malformed rows    : {r["bad_format"]}  <- TRUE errors')
        print(f'    Segmentation rows : {r["seg_format"]}  <- skipped silently (not errors)')
        print(f'    Invalid class IDs : {r["invalid_cls"]}')
        print(f'    Read errors       : {r["read_errors"]}')
        if r['examples']:
            print(f'    Sample issues:')
            for ex in r['examples']:
                print(f'      - {ex}')

    _print_split('train', train_r)
    _print_split('val',   val_r)

    total_true_errors = sum([
        train_r['missing_labels'], train_r['orphan_labels'],
        train_r['bad_format'], train_r['invalid_cls'], train_r['read_errors'],
        val_r['missing_labels'],  val_r['orphan_labels'],
        val_r['bad_format'],  val_r['invalid_cls'],  val_r['read_errors'],
    ])
    seg_rows = train_r['seg_format'] + val_r['seg_format']

    print(f'\n  Validation time : {elapsed:.1f}s')
    if seg_rows:
        print(f'  [INFO] {seg_rows} segmentation-format rows found and silently skipped.')
        print(f'         These are NOT errors -- YOLOv8 detection ignores them automatically.')
    if total_true_errors == 0:
        print('\n[OK] Dataset clean -- no true integrity issues found.')
        log.info('Dataset validation passed.')
    else:
        print(f'\n[WARNING] {total_true_errors} true issue(s) found.')
        log.warning(f'Dataset validation: {total_true_errors} issues, {seg_rows} seg rows ignored.')
    print(SEP)


---
## 6 · Exploratory Data Analysis <a id="eda"></a>
Class distribution charts and sample images with ground-truth bounding boxes. Set `RUN_EDA = False` in Cell 5 to skip.

In [ ]:
# ============================================================================
# CELL 8 -- EDA: Class Distribution + Sample Image Grid
# ============================================================================

def count_class_distribution(lbl_dir, num_classes):
    counts = np.zeros(num_classes, dtype=int)
    for lbl in lbl_dir.glob("*.txt"):
        try:
            for line in lbl.read_text().splitlines():
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if 0 <= cls_id < num_classes:
                        counts[cls_id] += 1
        except Exception:
            pass
    return counts

def draw_boxes_on_image(img_path, lbl_path, class_names, max_size=640):
    img = cv2.imread(str(img_path))
    if img is None:
        return np.zeros((320, 320, 3), dtype=np.uint8)
    h0, w0 = img.shape[:2]
    scale = min(max_size / max(h0, w0), 1.0)
    if scale < 1.0:
        img = cv2.resize(img, (int(w0 * scale), int(h0 * scale)),
                         interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]
    try:
        cmap = plt.colormaps["tab10"]
    except AttributeError:
        cmap = plt.cm.get_cmap("tab10")
    if lbl_path.exists():
        try:
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = max(0, int((cx - bw / 2) * w))
                y1 = max(0, int((cy - bh / 2) * h))
                x2 = min(w - 1, int((cx + bw / 2) * w))
                y2 = min(h - 1, int((cy + bh / 2) * h))
                color = tuple(int(c * 255) for c in cmap(cls_id % 10)[:3])[::-1]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(img, label, (x1, max(y1 - 4, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
        except Exception as exc:
            log.debug("Box drawing error %s: %s", img_path.name, exc)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

if not RUN_EDA:
    print("[INFO] EDA skipped (RUN_EDA=False).")
else:
    train_dist = count_class_distribution(TRAIN_LABELS, CFG.num_classes)
    val_dist   = count_class_distribution(VAL_LABELS,   CFG.num_classes)

    print("Class Distribution")
    print("=" * 70)
    print(f"{'Class':<25} {'Train':>8} {'Val':>8} {'Total':>8} {'%':>9}")
    print("-" * 70)
    total_all = int(train_dist.sum() + val_dist.sum())
    for i, name in enumerate(CFG.class_names):
        t, v = int(train_dist[i]), int(val_dist[i])
        pct  = (t + v) / total_all * 100 if total_all else 0
        print(f"  {name:<23} {t:>8,} {v:>8,} {t+v:>8,} {pct:>8.1f}%")
    print("-" * 70)
    print(f"  {'TOTAL':<23} {int(train_dist.sum()):>8,} {int(val_dist.sum()):>8,}"
          f" {total_all:>8,} {'100.0%':>9}")

    imb = float(train_dist.max()) / max(float(train_dist.min()), 1)
    print(f"\nImbalance ratio: {imb:.1f}x  ", end="")
    if imb > 10:
        print("[WARN] High imbalance -- consider class weighting.")
    elif imb > 3:
        print("[INFO] Moderate imbalance -- monitor per-class AP.")
    else:
        print("[OK] Reasonably balanced.")

    # Bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Class Distribution -- Pakistani Garbage Dataset",
                 fontsize=14, fontweight="bold")
    x  = np.arange(CFG.num_classes)
    bw = 0.35
    ax1.bar(x - bw/2, train_dist, bw, label="Train", color="#4C72B0", alpha=0.85)
    ax1.bar(x + bw/2, val_dist,   bw, label="Val",   color="#DD8452", alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(CFG.class_names, rotation=35, ha="right", fontsize=9)
    ax1.set_ylabel("Annotation count")
    ax1.set_title("Annotations per Class")
    ax1.legend()
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
    total_pc = train_dist + val_dist
    sidx = np.argsort(total_pc)[::-1]
    ax2.barh([CFG.class_names[i] for i in sidx], total_pc[sidx],
             color=sns.color_palette("muted", CFG.num_classes), alpha=0.9)
    ax2.set_xlabel("Total annotations")
    ax2.set_title("Ranked by Annotation Count")
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
    plt.tight_layout()
    eda_path = OUTPUT_DIR / "class_distribution.png"
    plt.savefig(eda_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"\n[OK] Chart saved: {eda_path}")

    # Sample image grid
    all_imgs = sorted(f for f in TRAIN_IMAGES.glob("*") if f.suffix.lower() in IMG_EXTENSIONS)
    samples  = all_imgs[:9]
    if samples:
        n_cols = 3
        n_rows = (len(samples) + n_cols - 1) // n_cols
        fig2, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows), squeeze=False)
        fig2.suptitle("Sample Training Images (Ground-Truth Boxes)",
                      fontsize=14, fontweight="bold")
        for idx, ax in enumerate(axes.flat):
            if idx < len(samples):
                ip = samples[idx]
                lp = TRAIN_LABELS / (ip.stem + ".txt")
                try:
                    ax.imshow(draw_boxes_on_image(ip, lp, CFG.class_names))
                    ax.set_title(ip.name[:30], fontsize=8)
                except Exception as exc:
                    ax.set_title(f"Error: {ip.name[:20]}", fontsize=7)
            ax.axis("off")
        plt.tight_layout()
        samples_path = OUTPUT_DIR / "sample_images.png"
        plt.savefig(samples_path, bbox_inches="tight")
        plt.show()
        plt.close(fig2)
        print(f"[OK] Sample grid saved: {samples_path}")
    else:
        print("[WARN] No training images found -- skipping sample grid.")

    log.info("EDA complete.")
print("[OK] Cell 8 done.")


---
## 7 · Training <a id="training"></a>
### 7a — Generate `data.yaml`
Always writes **absolute paths** — never reads path values from the dataset's own `data.yaml`, which may use relative paths that break on copy.

In [ ]:
# ============================================================================
# CELL 9 -- Generate data.yaml (Absolute Paths)
# ============================================================================
import yaml as _yaml

def generate_data_yaml(cfg, output_path):
    data = {
        "train": str(TRAIN_IMAGES.resolve()),
        "val":   str(VAL_IMAGES.resolve()),
        "test":  str(TEST_IMAGES.resolve()) if (TEST_IMAGES and TEST_IMAGES.exists())
                 else str(VAL_IMAGES.resolve()),
        "nc":    cfg.num_classes,
        "names": cfg.class_names,
    }
    with open(output_path, "w") as fh:
        _yaml.dump(data, fh, default_flow_style=False, allow_unicode=True)
    print(f"[OK] data.yaml -> {output_path}\n")
    with open(output_path) as fh:
        print(fh.read())
    # Pre-flight check
    ok = True
    for key in ("train", "val", "test"):
        p = Path(data[key])
        n = sum(1 for f in p.glob("*") if f.is_file()) if p.exists() else 0
        status = "[OK]" if p.exists() else "[MISSING]"
        if not p.exists():
            ok = False
        print(f"  {status} {key:<6}: {p}  [{n} files]")
    if not ok:
        raise FileNotFoundError("One or more split paths are missing. Re-run Cell 6.")
    return output_path

YAML_PATH = OUTPUT_DIR / "data.yaml"
yaml_path = generate_data_yaml(CFG, YAML_PATH)
log.info("data.yaml written: %s", yaml_path)


### 7b — YOLOv8 Training
Launches training with all hyperparameters from `CFG`. A per-epoch callback logs metrics to CSV. Resume mode is activated when `RESUME=True`.

In [ ]:
# ============================================================================
# CELL 10 — YOLOv8 Training  [FIXED VERSION — replace the original Cell 10]
# ============================================================================
#
# BUGS FIXED vs. original:
#
#  BUG 1 — "resume=RESUME" passed a bare True bool, not a path.
#           Ultralytics with resume=True (bool) calls _get_latest_run(),
#           which searches the DEFAULT ultralytics runs dir (~/.config/
#           Ultralytics/), NOT your custom project= path. So it never finds
#           last.pt and either crashes or starts fresh — looking like a reset.
#           FIX: always pass resume=str(last_pt) (explicit full path).
#
#  BUG 2 — No auto-resume detection. After a Kaggle 12-hr timeout, the next
#           Save Version starts with RESUME=False (the default), so training
#           restarted from epoch 1 every single time.
#           FIX: auto-detect last.pt at the known path and flip RESUME on.
#
#  BUG 3 — CSV history lost across sessions. OUTPUT_DIR gets a new timestamp
#           each run, so METRICS_CSV was always a fresh file. Previous epochs
#           (e.g. the 51 recorded before the timeout) were silently abandoned.
#           FIX: on resume, copy the most-recent old epoch_metrics.csv into
#           the new OUTPUT_DIR before training starts.
#
#  BUG 4 — _csv_fieldnames was set from the first epoch only. If
#           trainer.metrics happened to be empty on epoch 1 (Ultralytics can
#           return {} before the first full val pass), fieldnames were locked
#           to just ["epoch","timestamp"] and all metric columns were silently
#           dropped via extrasaction="ignore" for the rest of training.
#           FIX: seed fieldnames from the existing CSV when resuming;
#           otherwise defer header writing until we have a non-empty metrics
#           dict.
#
#  BUG 5 — 130 epochs × ~14 min/epoch ≈ 30 hrs >> 12-hr Kaggle limit.
#           Auto-resume (Bug 2 fix) now chains sessions automatically, but
#           a runtime warning is printed so you know how many sessions to
#           expect.
# ============================================================================

import csv as _csv

# ── Paths ─────────────────────────────────────────────────────────────────────
METRICS_CSV       = OUTPUT_DIR / "epoch_metrics.csv"
_RUNS_BASE        = WORKING_DIR / "runs"
_EXPECTED_LAST_PT = _RUNS_BASE / CFG.experiment_name / "weights" / "last.pt"

# ── BUG 2 FIX: Auto-resume detection ─────────────────────────────────────────
# If the user left RESUME=False but a checkpoint already exists from a prior
# (possibly timed-out) run, flip RESUME on automatically.
if not RESUME and _EXPECTED_LAST_PT.exists():
    print(f"[AUTO-RESUME] Checkpoint found at:")
    print(f"              {_EXPECTED_LAST_PT}")
    print(f"[AUTO-RESUME] Setting RESUME=True automatically.")
    print(f"              To start FRESH instead, delete weights/last.pt and re-run.\n")
    RESUME = True

# ── BUG 1 FIX: Resolve weights + explicit resume path ────────────────────────
# Never pass resume=True (bare bool). Always pass the explicit checkpoint path
# so Ultralytics doesn't fall back to _get_latest_run() and search the wrong
# directory.
if RESUME:
    # Priority: explicit RESUME_WEIGHTS > auto-detected last.pt
    if RESUME_WEIGHTS and Path(RESUME_WEIGHTS).exists():
        _ckpt_path = Path(RESUME_WEIGHTS)
    elif _EXPECTED_LAST_PT.exists():
        _ckpt_path = _EXPECTED_LAST_PT
    else:
        print("[WARN] RESUME=True but no checkpoint found at expected path.")
        print(f"       Expected: {_EXPECTED_LAST_PT}")
        print("       Starting training from pretrained weights instead.\n")
        _ckpt_path = None

    weights    = str(_ckpt_path) if _ckpt_path else CFG.model_name
    resume_arg = str(_ckpt_path) if _ckpt_path else False   # explicit path, not bare True
else:
    weights    = CFG.model_name
    resume_arg = False

# ── BUG 3 FIX: Restore previous epoch metrics on resume ──────────────────────
# OUTPUT_DIR is timestamped → METRICS_CSV is always a new file.
# Copy the most recent old epoch_metrics.csv into this session's OUTPUT_DIR
# so the full training history is preserved in one continuous file.
if RESUME and resume_arg:
    _all_old_csvs = sorted(
        (WORKING_DIR / "outputs").rglob("epoch_metrics.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for _oc in _all_old_csvs:
        if _oc != METRICS_CSV and _oc.stat().st_size > 10:
            shutil.copy2(_oc, METRICS_CSV)
            _n_prev = sum(1 for _ in open(METRICS_CSV)) - 1  # lines minus header
            print(f"[INFO] Restored {_n_prev} previous epoch records from {_oc.parent.name}/epoch_metrics.csv")
            break

# ── BUG 4 FIX: Robust CSV callback ───────────────────────────────────────────
# Seed fieldnames from the existing file (resume case) so we never drop columns.
# Otherwise defer until we get a real non-empty metrics dict from the trainer.
if METRICS_CSV.exists() and METRICS_CSV.stat().st_size > 0:
    with open(METRICS_CSV, newline="") as _f:
        _rdr = _csv.DictReader(_f)
        _csv_fieldnames     = list(_rdr.fieldnames or [])
        _csv_header_written = bool(_csv_fieldnames)
else:
    _csv_fieldnames     = []
    _csv_header_written = False

def _on_fit_epoch_end(trainer):
    """Append one row per epoch to METRICS_CSV. Thread-safe via file open/close."""
    global _csv_header_written, _csv_fieldnames

    raw   = trainer.metrics or {}
    epoch = trainer.epoch + 1          # trainer.epoch is 0-indexed

    # If metrics are still empty (Ultralytics sometimes returns {} on epoch 1
    # before the val pass completes), skip writing — we'll catch it next epoch.
    if not raw and not _csv_header_written:
        log.warning("Epoch %d: trainer.metrics is empty — skipping CSV row.", epoch)
        return

    row = {"epoch": epoch, "timestamp": datetime.now().isoformat(), **raw}

    with open(METRICS_CSV, "a", newline="") as f:
        if not _csv_header_written:
            # First epoch with real metrics — lock in the fieldnames.
            _csv_fieldnames     = list(row.keys())
            _csv_header_written = True
            _csv.DictWriter(f, fieldnames=_csv_fieldnames).writeheader()
        _csv.DictWriter(f, fieldnames=_csv_fieldnames,
                        extrasaction="ignore").writerow(row)

    if epoch == 1 or epoch % 5 == 0:
        log.info("Epoch %3d — %s", epoch,
                 {k: f"{v:.4f}" for k, v in raw.items() if isinstance(v, float)})

# ── Save config snapshot ──────────────────────────────────────────────────────
config_path = OUTPUT_DIR / "train_config.json"
with open(config_path, "w") as f:
    json.dump(asdict(CFG), f, indent=2)
print(f"[OK] Config saved: {config_path}")
print(f"[INFO] Weights   : {weights}")
print(f"[INFO] Resume    : {resume_arg}")

# ── BUG 5 WARNING: Epoch-time budget check ────────────────────────────────────
_MIN_PER_EPOCH_EST  = 14          # yolov8x on P100, conservative
_KAGGLE_BUDGET_HRS  = 11.5        # 12-hr limit with 30-min buffer
_MAX_SAFE_EPOCHS    = int(_KAGGLE_BUDGET_HRS * 60 / _MIN_PER_EPOCH_EST)
if not RESUME and CFG.epochs > _MAX_SAFE_EPOCHS:
    print(
        f"\n[BUDGET WARNING] {CFG.epochs} epochs × ~{_MIN_PER_EPOCH_EST} min/epoch "
        f"≈ {CFG.epochs * _MIN_PER_EPOCH_EST / 60:.0f} hrs  "
        f"(Kaggle limit ≈ {_KAGGLE_BUDGET_HRS} hrs)."
    )
    print(
        f"                 Auto-resume is now active — each new Save Version will\n"
        f"                 continue from the last checkpoint automatically.\n"
        f"                 You will need ~{CFG.epochs // _MAX_SAFE_EPOCHS + 1} sessions "
        f"to complete {CFG.epochs} epochs.\n"
    )

# ── Build train_args ──────────────────────────────────────────────────────────
train_args = dict(
    data         = str(yaml_path),
    epochs       = CFG.epochs,
    imgsz        = CFG.img_size,
    batch        = CFG.batch_size,
    lr0          = CFG.lr0,
    lrf          = CFG.lrf,
    momentum     = CFG.momentum,
    weight_decay = CFG.weight_decay,
    warmup_epochs= CFG.warmup_epochs,
    patience     = CFG.patience,
    workers      = CFG.workers,
    amp          = CFG.amp,
    cos_lr       = True,
    hsv_h        = CFG.hsv_h,
    hsv_s        = CFG.hsv_s,
    hsv_v        = CFG.hsv_v,
    degrees      = CFG.degrees,
    translate    = CFG.translate,
    scale        = CFG.scale,
    shear        = CFG.shear,
    perspective  = CFG.perspective,
    flipud       = CFG.flipud,
    fliplr       = CFG.fliplr,
    mosaic       = CFG.mosaic,
    mixup        = CFG.mixup,
    copy_paste   = CFG.copy_paste,
    project      = str(_RUNS_BASE),
    name         = CFG.experiment_name,
    exist_ok     = True,
    save         = True,
    save_period  = 10,
    plots        = True,
    verbose      = True,
    device       = 0 if CUDA_AVAILABLE else "cpu",
    resume       = resume_arg,   # ← EXPLICIT PATH or False (never bare True)
    seed         = SEED,
)

print("\n" + "=" * 70 + "\nSTARTING TRAINING\n" + "=" * 70)
print(f"  Model      : {weights}")
print(f"  Resume arg : {resume_arg}")
print(f"  Epochs     : {CFG.epochs}  (patience={CFG.patience})")
print(f"  Batch      : {CFG.batch_size}  |  imgsz={CFG.img_size}px")
print(f"  AMP        : {CFG.amp}  |  Device: {GPU_NAME if CUDA_AVAILABLE else 'CPU'}")
print(f"  Checkpoint : {_EXPECTED_LAST_PT}")
print("=" * 70)

# ── Create model and register callback ───────────────────────────────────────
model = YOLO(weights)

# Guard against duplicate registration if this cell is ever re-run
# interactively (does not affect Save Version runs, but prevents double-
# logging rows when debugging).
_existing = model.callbacks.get("on_fit_epoch_end", [])
if _on_fit_epoch_end not in _existing:
    model.add_callback("on_fit_epoch_end", _on_fit_epoch_end)

# ── Train ─────────────────────────────────────────────────────────────────────
t_start = time.time()
try:
    train_results = model.train(**train_args)
    elapsed = (time.time() - t_start) / 60
    log.info("Training complete in %.1f min.", elapsed)
    print(f"\n[OK] Training finished in {elapsed:.1f} min.")
    RUNS_DIR = Path(train_results.save_dir)
    print(f"[OK] Results directory: {RUNS_DIR}")
except Exception as exc:
    log.exception("Training failed: %s", exc)
    raise

---
## 8 · Evaluation <a id="eval"></a>
### 8a — Training Curves

In [ ]:
# ============================================================================
# CELL 11 -- Training Curves
# ============================================================================
SEP = "=" * 70

# Locate results.csv
results_csv = RUNS_DIR / "results.csv"
if not results_csv.exists():
    hits = sorted((WORKING_DIR / "runs").rglob("results.csv"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        results_csv = hits[0]
        RUNS_DIR    = results_csv.parent
        print(f"[INFO] results.csv found: {results_csv}")

if not results_csv.exists():
    print("[WARNING] results.csv not found -- training may not have completed.")
    log.warning("results.csv missing at %s", results_csv)
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"[OK] results.csv loaded -- {len(df)} epochs")

    metric_cols = [c for c in df.columns
                   if c.lower() != "epoch" and pd.api.types.is_numeric_dtype(df[c])]
    n_cols_p = 4
    n_rows_p = max(1, (len(metric_cols) + n_cols_p - 1) // n_cols_p)

    fig, axes = plt.subplots(n_rows_p, n_cols_p,
                             figsize=(5 * n_cols_p, 4 * n_rows_p), squeeze=False)
    fig.suptitle(f"Training Curves -- {CFG.experiment_name}",
                 fontsize=14, fontweight="bold")
    epoch_col = df["epoch"] if "epoch" in df.columns else range(len(df))

    for ax, col in zip(axes.flat, metric_cols):
        ax.plot(epoch_col, df[col], linewidth=1.8, color="#4C72B0")
        ax.set_title(col, fontsize=10)
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.4)
    for ax in list(axes.flat)[len(metric_cols):]:
        ax.set_visible(False)

    plt.tight_layout()
    curves_path = OUTPUT_DIR / "training_curves.png"
    plt.savefig(curves_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"\n[OK] Training curves: {curves_path}")

    print("\nFinal Epoch Metrics:")
    print("-" * 55)
    last = df.iloc[-1]
    for col in metric_cols:
        try:
            print(f"  {col:<40}: {float(last[col]):.4f}")
        except (ValueError, TypeError):
            pass


### 8b — Validation Metrics & Per-Class AP

In [ ]:
# ============================================================================
# CELL 12 -- Model Evaluation on Validation Split
# ============================================================================
SEP = "=" * 70

# Locate best checkpoint
if not (RUNS_DIR / "weights").exists():
    hits = sorted((WORKING_DIR / "runs").rglob("best.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        RUNS_DIR = hits[0].parent.parent
        print(f"[INFO] RUNS_DIR updated: {RUNS_DIR}")

best_pt = RUNS_DIR / "weights" / "best.pt"
last_pt = RUNS_DIR / "weights" / "last.pt"

if best_pt.exists():
    ckpt_path = best_pt
    print(f"[OK] Loading best.pt: {ckpt_path}")
elif last_pt.exists():
    ckpt_path = last_pt
    print(f"[WARN] best.pt absent -- using last.pt: {ckpt_path}")
else:
    candidates = sorted((WORKING_DIR / "runs").rglob("best.pt"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        ckpt_path = candidates[0]
    else:
        raise FileNotFoundError("No checkpoint found. Did training complete?")

eval_model = YOLO(str(ckpt_path))

print("\nRunning validation ...")
val_results = eval_model.val(
    data=str(yaml_path), split="val", imgsz=CFG.img_size,
    device=0 if CUDA_AVAILABLE else "cpu", verbose=False,
)

metrics = {
    "mAP@0.5":      float(val_results.box.map50),
    "mAP@0.5:0.95": float(val_results.box.map),
    "Precision":    float(val_results.box.mp),
    "Recall":       float(val_results.box.mr),
}
metrics["F1"] = (
    2 * metrics["Precision"] * metrics["Recall"]
    / (metrics["Precision"] + metrics["Recall"] + 1e-8)
)

print("\n" + SEP + "\nVALIDATION METRICS\n" + SEP)
for k, v in metrics.items():
    bar = chr(9608) * int(v * 25)
    print(f"  {k:<20}: {v:.4f}  {bar}")

ap50_per_class = []
if hasattr(val_results.box, "ap50") and val_results.box.ap50 is not None:
    ap50_per_class = val_results.box.ap50.tolist()
    print("\nPer-Class AP@0.5:")
    print("-" * 55)
    for cls_name, ap in sorted(zip(CFG.class_names, ap50_per_class),
                                key=lambda x: x[1], reverse=True):
        bar = chr(9608) * int(ap * 25)
        print(f"  {cls_name:<25}: {ap:.4f}  {bar}")

    fig, ax = plt.subplots(figsize=(10, 5))
    sorted_pairs = sorted(zip(CFG.class_names, ap50_per_class), key=lambda x: x[1])
    names_s = [p[0] for p in sorted_pairs]
    aps_s   = [p[1] for p in sorted_pairs]
    colors  = ["#d9534f" if a < 0.5 else "#f0ad4e" if a < 0.75 else "#5cb85c"
               for a in aps_s]
    ax.barh(names_s, aps_s, color=colors, alpha=0.85)
    ax.axvline(0.50, ls="--", color="red",    lw=1, label="0.50")
    ax.axvline(0.75, ls="--", color="orange", lw=1, label="0.75")
    ax.set_xlabel("AP@0.5")
    ax.set_title("Per-Class Average Precision @ IoU 0.5", fontweight="bold")
    ax.legend()
    ax.set_xlim(0, 1)
    plt.tight_layout()
    ap_path = OUTPUT_DIR / "per_class_ap.png"
    plt.savefig(ap_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"\n[OK] Per-class AP chart: {ap_path}")

metrics_json = OUTPUT_DIR / "validation_metrics.json"
payload = {
    "model_checkpoint": str(ckpt_path),
    "timestamp":        datetime.now().isoformat(),
    "config":           asdict(CFG),
    "aggregate_metrics": metrics,
    "per_class_ap50":   dict(zip(CFG.class_names, ap50_per_class)),
}
with open(metrics_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"[OK] Metrics JSON: {metrics_json}")
log.info("Val mAP@0.5=%.4f  mAP@0.5:0.95=%.4f",
         metrics["mAP@0.5"], metrics["mAP@0.5:0.95"])
print(SEP)


---
## 9 · Test Set Evaluation <a id="test"></a>
Optional — only runs if `RUN_TEST_EVAL = True` (Cell 5) and a `test/` split was found.

In [ ]:
# ============================================================================
# CELL 13 -- Test Set Evaluation (Optional)
# ============================================================================
SEP = "=" * 70

if not RUN_TEST_EVAL:
    print("[INFO] Test evaluation skipped (RUN_TEST_EVAL=False).")
elif TEST_IMAGES is None or not TEST_IMAGES.exists():
    print("[INFO] No test split found -- skipping.")
else:
    print("\n" + SEP + "\nTEST SET EVALUATION\n" + SEP)
    try:
        test_results = eval_model.val(
            data=str(yaml_path), split="test", imgsz=CFG.img_size,
            device=0 if CUDA_AVAILABLE else "cpu", verbose=False,
        )
        test_metrics = {
            "mAP@0.5":      float(test_results.box.map50),
            "mAP@0.5:0.95": float(test_results.box.map),
            "Precision":    float(test_results.box.mp),
            "Recall":       float(test_results.box.mr),
        }
        test_metrics["F1"] = (
            2 * test_metrics["Precision"] * test_metrics["Recall"]
            / (test_metrics["Precision"] + test_metrics["Recall"] + 1e-8)
        )
        print("\nTest Metrics:")
        for k, v in test_metrics.items():
            bar = chr(9608) * int(v * 25)
            print(f"  {k:<20}: {v:.4f}  {bar}")
        test_json = OUTPUT_DIR / "test_metrics.json"
        with open(test_json, "w") as f:
            json.dump({"test_metrics": test_metrics}, f, indent=2)
        print(f"\n[OK] Test metrics: {test_json}")
        log.info("Test mAP@0.5=%.4f", test_metrics["mAP@0.5"])
    except Exception as exc:
        log.error("Test evaluation failed: %s", exc)
        print(f"[WARN] Test evaluation failed: {exc}")
    print(SEP)


---
## 10 · Export & Deployment <a id="export"></a>
Exports to **ONNX** (always). Attempts **TensorRT** engine export (GPU only, optional). Runs an ONNX Runtime inference smoke test to verify the exported graph.

In [ ]:
# ============================================================================
# CELL 14 -- ONNX Export + TensorRT (optional) + ONNX Runtime Smoke Test
# ============================================================================
SEP = "=" * 70
print("\n" + SEP + "\nMODEL EXPORT\n" + SEP)

dest_onnx = None

# -- ONNX -------------------------------------------------------------------
try:
    print("Exporting to ONNX (opset 17, simplified) ...")
    onnx_raw = eval_model.export(
        format="onnx", imgsz=CFG.img_size,
        simplify=True, dynamic=False, half=False, opset=17,
    )
    if onnx_raw is None:
        raise RuntimeError("export() returned None.")
    onnx_path = Path(str(onnx_raw))
    if not onnx_path.exists():
        raise FileNotFoundError(f"Exported file not found: {onnx_path}")

    dest_onnx = OUTPUT_DIR / "best.onnx"
    shutil.copy2(onnx_path, dest_onnx)
    mb = dest_onnx.stat().st_size / 1024**2
    print(f"[OK] ONNX saved: {dest_onnx}  ({mb:.1f} MB)")
    log.info("ONNX export OK -- %.1f MB", mb)

    try:
        import onnx as _onnx
        _onnx.checker.check_model(str(dest_onnx))
        print("[OK] ONNX graph validation passed.")
    except ImportError:
        print("[INFO] onnx not installed -- skipping graph check.")
    except Exception as val_err:
        print(f"[WARN] ONNX graph check: {val_err}")
except Exception as exc:
    log.error("ONNX export failed: %s", exc)
    print(f"[WARN] ONNX export failed: {exc}")
    print("       best.pt is still available for PyTorch inference.")

# -- TensorRT (optional) ----------------------------------------------------
if CUDA_AVAILABLE:
    print("\nAttempting TensorRT export ...")
    try:
        trt_raw = eval_model.export(
            format="engine", imgsz=CFG.img_size, half=True, device=0,
        )
        if trt_raw is not None:
            trt_path = Path(str(trt_raw))
            if trt_path.exists():
                dest_trt = OUTPUT_DIR / "best.engine"
                shutil.copy2(trt_path, dest_trt)
                mb_trt = dest_trt.stat().st_size / 1024**2
                print(f"[OK] TensorRT engine: {dest_trt}  ({mb_trt:.1f} MB)")
            else:
                print("[WARN] TensorRT export path not found.")
        else:
            print("[WARN] TensorRT export returned None.")
    except Exception as trt_exc:
        print(f"[INFO] TensorRT skipped: {trt_exc}")
        print("       Normal if TensorRT is not installed.")
else:
    print("\n[INFO] TensorRT requires a GPU -- skipping.")

# -- ONNX Runtime Smoke Test ------------------------------------------------
print("\n" + "-" * 55)
print("ONNX Runtime Inference Smoke Test")
print("-" * 55)

if dest_onnx is None or not dest_onnx.exists():
    print("[SKIP] No ONNX file to test.")
else:
    try:
        import onnxruntime as ort
        sample_imgs = [f for f in VAL_IMAGES.glob("*")
                       if f.suffix.lower() in IMG_EXTENSIONS]
        if not sample_imgs:
            raise FileNotFoundError("No val images for smoke test.")

        test_img = sample_imgs[0]
        raw  = cv2.imread(str(test_img))
        if raw is None:
            raise ValueError(f"Could not read: {test_img}")
        resz = cv2.resize(raw, (CFG.img_size, CFG.img_size))
        blob = resz[:, :, ::-1].transpose(2, 0, 1).astype(np.float32) / 255.0
        blob = blob[np.newaxis, ...]   # [1, 3, H, W]

        sess     = ort.InferenceSession(str(dest_onnx))
        inp_name = sess.get_inputs()[0].name
        outputs  = sess.run(None, {inp_name: blob})

        print(f"[OK] ONNX Runtime session ready.")
        print(f"     Input  : {inp_name}  shape={blob.shape}")
        print(f"     Output : {len(outputs)} tensor(s), shape={outputs[0].shape}")
        print(f"     Image  : {test_img.name}")
        print("[OK] ONNX Runtime smoke test PASSED.")
        log.info("ONNX Runtime smoke test passed for %s", test_img.name)
    except ImportError:
        print("[INFO] onnxruntime not installed -- skipping smoke test.")
    except Exception as rt_exc:
        print(f"[WARN] ONNX smoke test failed: {rt_exc}")
        log.warning("ONNX smoke test: %s", rt_exc)
print(SEP)


---
## 11 · Artifact Collection <a id="artifacts"></a>
Copies all training artifacts from the Ultralytics `runs/` folder into the timestamped output directory.

In [ ]:
# ============================================================================
# CELL 15 -- Collect & Organise Training Artifacts
# ============================================================================
SEP = "=" * 70
print("\n" + SEP + "\nARTIFACT COLLECTION\n" + SEP)

if not (RUNS_DIR / "weights").exists():
    hits = sorted((WORKING_DIR / "runs").rglob("best.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        RUNS_DIR = hits[0].parent.parent
        print(f"[INFO] RUNS_DIR resolved: {RUNS_DIR}")

artifact_map = {
    "best.pt":               RUNS_DIR / "weights" / "best.pt",
    "last.pt":               RUNS_DIR / "weights" / "last.pt",
    "results.png":           RUNS_DIR / "results.png",
    "confusion_matrix.png":  RUNS_DIR / "confusion_matrix.png",
    "confusion_matrix_normalized.png": RUNS_DIR / "confusion_matrix_normalized.png",
    "F1_curve.png":          RUNS_DIR / "F1_curve.png",
    "PR_curve.png":          RUNS_DIR / "PR_curve.png",
    "P_curve.png":           RUNS_DIR / "P_curve.png",
    "R_curve.png":           RUNS_DIR / "R_curve.png",
    "labels.jpg":            RUNS_DIR / "labels.jpg",
    "results.csv":           RUNS_DIR / "results.csv",
}

saved = skipped = 0
for dest_name, src in artifact_map.items():
    if src.exists():
        dest = OUTPUT_DIR / dest_name
        shutil.copy2(src, dest)
        mb = dest.stat().st_size / 1024**2
        print(f"  [OK] {dest_name:<45} ({mb:.2f} MB)")
        saved += 1
    else:
        print(f"  [--] {dest_name:<45} not found")
        skipped += 1

print(f"\n  Copied {saved} / {len(artifact_map)} artifacts")
print(f"  Output dir: {OUTPUT_DIR}")
print(SEP)


---
## 12 · Final Summary & Download Instructions <a id="summary"></a>

In [ ]:
# ============================================================================
# CELL 16 -- Final Summary
# ============================================================================
SEP = "=" * 70
print("\n" + SEP + "\nFINAL TRAINING SUMMARY\n" + SEP)

_m = metrics if "metrics" in dir() and isinstance(metrics, dict) else {}

summary_lines = [
    f"Project       : {CFG.project_name}",
    f"Experiment    : {CFG.experiment_name}",
    f"Run timestamp : {RUN_TS}",
    f"Run date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Platform      : {'Kaggle' if IN_KAGGLE else 'Local'}",
    f"GPU           : {GPU_NAME}",
    f"VRAM          : {VRAM_GB:.2f} GB",
    "",
    "-- Dataset ---------------------------------------------------",
    f"  Train  : {count_images(TRAIN_IMAGES):,} images",
    f"  Val    : {count_images(VAL_IMAGES):,} images",
    f"  Test   : {count_images(TEST_IMAGES) if TEST_IMAGES else 'N/A'} images",
    f"  Classes: {CFG.num_classes}  ->  {chr(44).join(CFG.class_names)}",
    "",
    "-- Model -----------------------------------------------------",
    f"  Base model   : {CFG.model_name}",
    f"  Image size   : {CFG.img_size}px",
    f"  Batch size   : {CFG.batch_size}",
    f"  Epochs       : {CFG.epochs}  (patience={CFG.patience})",
    f"  AMP          : {CFG.amp}",
    f"  Seed         : {SEED}",
    "",
    "-- Validation Performance ------------------------------------",
    f"  mAP@0.5      : {_m.get('mAP@0.5',      float('nan')):.4f}",
    f"  mAP@0.5:0.95 : {_m.get('mAP@0.5:0.95', float('nan')):.4f}",
    f"  Precision    : {_m.get('Precision',     float('nan')):.4f}",
    f"  Recall       : {_m.get('Recall',        float('nan')):.4f}",
    f"  F1           : {_m.get('F1',            float('nan')):.4f}",
]

for line in summary_lines:
    print(line)

print("\n-- Output Files -----------------------------------------------")
total_mb = 0.0
for fpath in sorted(OUTPUT_DIR.glob("*")):
    if fpath.is_file():
        mb = fpath.stat().st_size / 1024**2
        total_mb += mb
        print(f"  * {fpath.name:<48} {mb:>7.2f} MB")
print(f"\n  Total: {total_mb:.2f} MB  in  {OUTPUT_DIR}")

summary_path = OUTPUT_DIR / "TRAINING_SUMMARY.txt"
usage_block = """
-- Download from Kaggle ----------------------------------------
  1. Click  Save Version  ->  Save & Run All
  2. Open   Output  tab  ->  Download All

-- Load for PyTorch inference ----------------------------------
  from ultralytics import YOLO
  model = YOLO("best.pt")
  results = model.predict("photo.jpg", conf=0.25)
  results[0].show()

-- ONNX Runtime inference -------------------------------------
  import onnxruntime as ort, numpy as np, cv2
  sess  = ort.InferenceSession("best.onnx")
  inp   = sess.get_inputs()[0].name
  img   = cv2.resize(cv2.imread("photo.jpg"), (640, 640))
  blob  = img[:,:,::-1].transpose(2,0,1)[None].astype(np.float32)/255
  out   = sess.run(None, {inp: blob})
  print(out[0].shape)
"""
with open(summary_path, "w") as f:
    f.write("\n".join(summary_lines))
    f.write(usage_block)
print(f"\n[OK] Summary saved: {summary_path}")

print("\n" + SEP)
print("[DONE] PIPELINE COMPLETE")
print(f"       Outputs -> {OUTPUT_DIR}")
print(SEP)
log.info("Pipeline complete. All outputs in %s", OUTPUT_DIR)
